# TAAS 전국·서울 교통사고 발생 추이 정제

노트북을 프로젝트 루트에 두고, 첨부한 엑셀 두 개를 `data/raw`에 원본 파일명 그대로 넣어. 위에서부터 실행하면 2020~2025년 전국·서울의 사고 건수가 CSV로 저장돼.

Python 커널에 `openpyxl`이 필요해. 설치가 필요하면 노트북 셀에서 `%pip install openpyxl`을 실행해.


In [ ]:
from pathlib import Path

from openpyxl import load_workbook

sources = [
    Path("data/raw/교통사고통계_2020.xlsx"),
    Path("data/raw/교통사고통계_2021-2025.xlsx"),
]
output = Path("data/15_전국서울_교통사고_추이.csv")

def read_taas_workbook(source):
    if not source.is_file():
        raise FileNotFoundError(f"원본 엑셀 경로를 확인해 줘. {source.resolve()}")

    workbook = load_workbook(source, read_only=True, data_only=True)
    try:
        sheet = workbook.active
        # TAAS 엑셀에 실제 표보다 작은 시트 크기가 표시될 수 있어.
        sheet.reset_dimensions()
        header, *rows = sheet.values
    finally:
        workbook.close()

    print(f"원본 {source.name} · 헤더 제외 {len(rows):,}행")
    return [str(value).strip() for value in header], rows


## 전국과 서울의 사고 건수만 추출

전국은 `합계 / 합계`, 서울은 `서울 / 합계` 행이야. 두 지역 모두 `사고[건]`만 선택해. 자치구나 사망·부상 건수를 합산할 필요가 없어.

In [ ]:
def clean_text(value):
    return "" if value is None else str(value).strip()

def accident_count(value):
    if isinstance(value, (int, float)) and value >= 0 and int(value) == value:
        return int(value)
    cleaned = clean_text(value).replace(",", "")
    if cleaned.isdecimal():
        return int(cleaned)
    raise ValueError(f"유효한 사고 건수가 아니야. {value!r}")

def extract_accidents(header, rows):
    region_col = header.index("시도")
    district_col = header.index("시군구")
    measure_col = header.index("연도")
    year_columns = sorted(
        (int(label), index)
        for index, label in enumerate(header)
        if len(label) == 4 and label.isdecimal()
    )
    years = [year for year, _ in year_columns]
    if not years or len(years) != len(set(years)):
        raise ValueError("연도 열이 없거나 중복돼 있어.")

    totals = {}
    for name, region in [("전국", "합계"), ("서울", "서울")]:
        matching = [
            row for row in rows
            if clean_text(row[region_col]) == region
            and clean_text(row[district_col]) == "합계"
            and clean_text(row[measure_col]) == "사고[건]"
        ]
        if len(matching) != 1:
            raise ValueError(f"{name} 사고 합계 행이 1개여야 해. 찾은 행 수 {len(matching)}")
        row = matching[0]
        totals[name] = [accident_count(row[index]) for _, index in year_columns]

    records = list(zip(years, totals["전국"], totals["서울"]))
    if any(seoul > nationwide for _, nationwide, seoul in records):
        raise ValueError("서울 사고 건수가 전국보다 큰 연도가 있어.")
    return records

records_by_year = {}
for source in sources:
    header, rows = read_taas_workbook(source)
    for year, nationwide, seoul in extract_accidents(header, rows):
        if year in records_by_year:
            raise ValueError(f"{year}년 자료가 두 파일에 중복돼 있어.")
        records_by_year[year] = (nationwide, seoul)

expected_years = set(range(2020, 2026))
if set(records_by_year) != expected_years:
    missing = sorted(expected_years - records_by_year.keys())
    extra = sorted(records_by_year.keys() - expected_years)
    raise ValueError(f"연도 범위를 확인해 줘. 누락 {missing}, 범위 밖 {extra}")

records = [(year, *counts) for year, counts in sorted(records_by_year.items())]
print("연도 | 전국(건) | 서울(건)")
for year, nationwide, seoul in records:
    print(f"{year} | {nationwide:,} | {seoul:,}")


## 차트용 CSV 저장

정상 결과는 데이터 6행이야. 첫해는 2020년 전국 209,654건·서울 35,227건, 마지막 해는 2025년 전국 193,889건·서울 34,309건이야. 두 엑셀 원본은 그대로 보관해.


In [ ]:
import csv

output.parent.mkdir(parents=True, exist_ok=True)
with output.open("w", encoding="utf-8-sig", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["year", "nationwide_accidents", "seoul_accidents"])
    writer.writerows(records)

print(f"{len(records)}개 연도 저장 완료 · {output}")
